# 04 — Populate SQLite database from cleaned CSVs

This notebook loads the cleaned CSVs from `data/processed/` into the SQLite database.

**Order matters:**
1. Ensure the schema exists (run `02_build_sqlite_db.ipynb` or create the DB via CLI first).
2. Load **continents** first so `country_id` is assigned.
3. Build a **country name → country_id** mapping.
4. Load **economy**, **population**, **education**, **education_quality** using that mapping.

We then run basic SQL validation checks.

## Setup: paths and database connection

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROJECT_ROOT / "world_indicators.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
print("Connected to:", DB_PATH)

## Clear existing data (optional)

If you are re-running this notebook and want a fresh load, delete rows in dependency order: fact tables first, then continents. **Run this cell only when you intend to repopulate.**

In [ ]:
# Uncomment to reset data:
# conn.execute("DELETE FROM economy")
# conn.execute("DELETE FROM population")
# conn.execute("DELETE FROM education")
# conn.execute("DELETE FROM education_quality")
# conn.execute("DELETE FROM continents")
# conn.commit()
# print("Tables cleared.")

## 1. Load continents

Insert from `data/processed/continents.csv`. The table has `country_id INTEGER PRIMARY KEY AUTOINCREMENT`, so we only insert `(country, continent, subregion)`.

In [ ]:
continents_df = pd.read_csv(PROCESSED_DIR / "continents.csv")
print("Rows to insert:", len(continents_df))
continents_df.head()

In [ ]:
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM continents")
n_before = cur.fetchone()[0]
if n_before == 0:
    cur.executemany(
        "INSERT INTO continents (country, continent, subregion) VALUES (?, ?, ?)",
        continents_df[["country", "continent", "subregion"]].itertuples(index=False, name=None),
    )
    conn.commit()
    print("Inserted", cur.rowcount, "rows into continents.")
else:
    print("Continents already has", n_before, "rows; skipping insert (clear table first to repopulate).")

## 2. Build country → country_id mapping

We need this to insert into economy, population, education, and education_quality (they store `country_id`, not the name).

In [ ]:
country_to_id = pd.read_sql("SELECT country_id, country FROM continents", conn).set_index("country")["country_id"].to_dict()
print("Mapping size:", len(country_to_id))
print("Examples:", list(country_to_id.items())[:5])

## 3. Load data into database tables

Read `economy.csv`, add `country_id` from the mapping, then insert only the columns that belong in the `economy` table. Skip rows whose country is not in `continents` (e.g. if a CSV has an extra row).

In [ ]:
economy_df = pd.read_csv(PROCESSED_DIR / "economy.csv")
economy_df["country_id"] = economy_df["country"].map(country_to_id)
economy_df = economy_df.dropna(subset=["country_id"])
economy_df["country_id"] = economy_df["country_id"].astype(int)
print("Rows to insert (after matching to continents):", len(economy_df))
economy_df.head()

In [ ]:
cur = conn.cursor()
cur.executemany(
    """INSERT INTO economy (country_id, population_2020, gdp_per_capita_2020_usd, avg_gov_exp_edu_gdp_pct_20y)
       VALUES (?, ?, ?, ?)""",
    economy_df[["country_id", "population_2020", "gdp_per_capita_2020_usd", "avg_gov_exp_edu_gdp_pct_20y"]].itertuples(index=False, name=None),
)
conn.commit()
print("Inserted", cur.rowcount, "rows into economy.")

Read `population.csv`, add `country_id` from the mapping, drop rows with missing `country_id`, then `INSERT` into `population (country_id, population_2020, urban_pop_pct, gdp_per_capita_2020_usd)`.

In [ ]:
population_df = pd.read_csv(PROCESSED_DIR / "population.csv")
population_df["country_id"] = population_df["country"].map(country_to_id)
population_df = population_df.dropna(subset=["country_id"])
population_df["country_id"] = population_df["country_id"].astype(int)
print("Rows to insert (after matching to continents):", len(population_df))
population_df.head()

In [ ]:
cur = conn.cursor()
cur.executemany(
    """
    INSERT INTO population (country_id, population_2020, urban_pop_pct, gdp_per_capita_2020_usd)
    VALUES (?, ?, ?, ?)""",
    population_df[["country_id", "population_2020", "urban_pop_pct", "gdp_per_capita_2020_usd"]].
    itertuples(index=False, name=None),
)
conn.commit()
print("Inserted", cur.rowcount, "rows into population.")

Read `education.csv`, map `country` → `country_id`, then insert into `education (country_id, avg_primary_enrollment_pct, avg_primary_completion_pct, avg_secondary_enrollment_pct, avg_secondary_completion_pct, avg_tertiary_enrollment_pct, avg_tertiary_completion_pct)`.

In [ ]:
education_df = pd.read_csv(PROCESSED_DIR / "education.csv")
education_df["country_id"] = education_df["country"].map(country_to_id)
education_df = education_df.dropna(subset=["country_id"])
education_df["country_id"] = education_df["country_id"].astype(int)
print("Rows to insert (after matching to continents):", len(education_df))
education_df.head()

In [ ]:
cur = conn.cursor()
cur.executemany(
    """
    INSERT INTO education (country_id, avg_primary_enrollment_pct, avg_primary_completion_pct, avg_secondary_enrollment_pct, avg_secondary_completion_pct, avg_tertiary_enrollment_pct, avg_tertiary_completion_pct)
    VALUES (?, ?, ?, ?, ?, ?, ?)""",
    education_df[["country_id", "avg_primary_enrollment_pct", "avg_primary_completion_pct", "avg_secondary_enrollment_pct", "avg_secondary_completion_pct", "avg_tertiary_enrollment_pct", "avg_tertiary_completion_pct"]].
    itertuples(index=False, name=None),
)
conn.commit()
print("Inserted", cur.rowcount, "rows into education.")

Read `education_quality.csv`, map `country` → `country_id`, then insert into `education_quality (country_id, avg_pisa_reading, avg_pisa_mathematics, avg_pisa_science)`.

In [ ]:
education_quality_df = pd.read_csv(PROCESSED_DIR / "education_quality.csv")
education_quality_df["country_id"] = education_quality_df["country"].map(country_to_id)
education_quality_df = education_quality_df.dropna(subset=["country_id"])
education_quality_df["country_id"] = education_quality_df["country_id"].astype(int)
print("Rows to insert (after matching to continents):", len(education_quality_df))
education_quality_df.head()

In [ ]:
cur = conn.cursor()
cur.executemany(
    """
    INSERT INTO education_quality (country_id, avg_pisa_reading, avg_pisa_mathematics, avg_pisa_science)
    VALUES (?, ?, ?, ?)""",
    education_quality_df[["country_id", "avg_pisa_reading", "avg_pisa_mathematics", "avg_pisa_science"]].
    itertuples(index=False, name=None),
)
conn.commit()
print("Inserted", cur.rowcount, "rows into education_quality.")

## 4. Basic SQL validation

Run a few checks to ensure row counts and joins look correct.

In [ ]:
for table in ["continents", "economy", "population", "education", "education_quality"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {n} rows")

In [ ]:
# Sample join: country name + GDP (should have one row per country in economy)
sample = pd.read_sql("""
    SELECT c.country, c.continent, e.gdp_per_capita_2020_usd
    FROM continents c
    JOIN economy e ON e.country_id = c.country_id
    ORDER BY e.gdp_per_capita_2020_usd DESC
    LIMIT 10
""", conn)
print("Top 5 by GDP per capita:")
display(sample)

In [ ]:
conn.close()
print("Connection closed.")